<a href="https://www.kaggle.com/code/muhammadmubeenxt/flooding-experiment-2-march-13?scriptVersionId=303433619" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Libraries

In [ ]:
# Run this cell first to install all required packages
!pip install pystac-client planetary-computer stackstac pyproj rioxarray dask odc-stac -q
!pip install rasterio python-dotenv requests segmentation-models-pytorch

# Multiple imagery download


In [ ]:
import os
import time
import json
import logging
import pandas as pd
import numpy as np
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import xarray as xr
from requests.exceptions import ReadTimeout

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("bulk_download.log"),
        logging.StreamHandler()
    ]
)

# --- The Core Download Function ---
def fetch_fusion_ready_imagery(lat, lon, radius_km, target_date, event_label, output_dir):
    """
    Fetches, co-registers, and stacks S1 (RTC) and S2 (L2A) into a float32 GeoTIFF.
    """
    date_buffer = pd.to_timedelta("3 days")
    start_date = (pd.to_datetime(target_date) - date_buffer).strftime('%Y-%m-%d')
    end_date = (pd.to_datetime(target_date) + date_buffer).strftime('%Y-%m-%d')
    time_range = f"{start_date}/{end_date}"
    
    # Approx km to degrees
    degree_buffer = radius_km / 111.0
    bbox = [lon - degree_buffer, lat - degree_buffer, lon + degree_buffer, lat + degree_buffer]

    logging.info(f"Processing: {event_label} | Date: {target_date} | BBOX: {bbox}")

    try:
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=planetary_computer.sign_inplace,
        )

        # Sentinel-1 RTC Search (Primary)
        s1_search = catalog.search(
            collections=["sentinel-1-rtc"], bbox=bbox, datetime=time_range
        )
        s1_items = list(s1_search.items())
        if not s1_items:
            logging.error(f"[{event_label}] No S1 RTC data found. Skipping.")
            return False

        # Sentinel-2 L2A Search (Secondary/Contextual)
        s2_search = catalog.search(
            collections=["sentinel-2-l2a"], bbox=bbox, datetime=time_range,
            query={"eo:cloud_cover": {"lt": 50}} # Allow some clouds during disasters
        )
        s2_items = list(s2_search.items())

        shared_crs = "EPSG:3857"
        shared_resolution = 10

        logging.info(f"[{event_label}] Loading SAR array...")
        s1_ds = odc.stac.load(
            s1_items, bands=["vv", "vh"], bbox=bbox, crs=shared_crs,
            resolution=shared_resolution, patch_url=planetary_computer.sign
        ).median(dim="time")

        if s2_items:
            logging.info(f"[{event_label}] Loading Optical array...")
            s2_ds = odc.stac.load(
                s2_items, bands=["B02", "B03", "B04", "B08", "SCL"], bbox=bbox,
                crs=shared_crs, resolution=shared_resolution, patch_url=planetary_computer.sign
            ).median(dim="time")
            # Force merge despite variable dimension metadata
            fused_ds = xr.merge([s1_ds, s2_ds], compat="override")
        else:
            logging.warning(f"[{event_label}] No Optical data. Stacking SAR only.")
            fused_ds = s1_ds

        logging.info(f"[{event_label}] Formatting and saving to disk...")
        # Cast to float32 to prevent numpy/rioxarray type collisions
        fused_da = fused_ds.to_array(dim="band").astype("float32")
        fused_da.rio.write_nodata(np.nan, inplace=True)
        fused_da.rio.write_crs(shared_crs, inplace=True)

        safe_label = event_label.replace(" ", "_").upper()
        safe_date = target_date.replace("-", "")
        filename = f"MANZAR_{safe_label}_S1S2_{safe_date}_{lat:.4f}_{lon:.4f}.tif"
        filepath = os.path.join(output_dir, filename)

        fused_da.rio.to_raster(filepath, compress="LZW", tiled=True)
        logging.info(f"[{event_label}] SUCCESS. Saved: {filename}")
        return True

    except ReadTimeout:
         logging.error(f"[{event_label}] STAC API Timeout. Server under load.")
         return False
    except Exception as e:
        logging.error(f"[{event_label}] FAILED: {str(e)}")
        return False

# --- The Bulk Execution Manager ---
def run_bulk_download():
    output_dir = "./data/raw_flood_stacks"
    os.makedirs(output_dir, exist_ok=True)

    # 25 Major Flood Events (2016-Present) globally
    flood_events = [
        # Pakistan
        #{"label": "Sindh_Pakistan_Flood_1", "lat": 26.2, "lon": 68.0, "date": "2022-08-25", "radius": 15},
        #{"label": "Sindh_Pakistan_Flood_2", "lat": 27.5, "lon": 68.5, "date": "2022-08-26", "radius": 15},
        #{"label": "KPK_Pakistan_Flood", "lat": 34.0, "lon": 71.9, "date": "2022-08-28", "radius": 15},
        #{"label": "Balochistan_Pakistan_Flood", "lat": 28.5, "lon": 66.0, "date": "2022-07-25", "radius": 15},
        #{"label": "Punjab_Pakistan_Flood", "lat": 30.2, "lon": 70.9, "date": "2023-08-15", "radius": 15},
        
        # South Asia
        #{"label": "Assam_India_Flood", "lat": 26.2, "lon": 92.0, "date": "2020-07-15", "radius": 15},
        #{"label": "Kerala_India_Flood", "lat": 9.9, "lon": 76.2, "date": "2018-08-16", "radius": 15},
        #{"label": "Sylhet_Bangladesh_Flood", "lat": 24.9, "lon": 91.8, "date": "2022-06-18", "radius": 15},
        #{"label": "Kathmandu_Nepal_Flood", "lat": 27.7, "lon": 85.3, "date": "2017-08-12", "radius": 10},
        
        # Asia Pacific
        #{"label": "Henan_China_Flood", "lat": 34.7, "lon": 113.6, "date": "2021-07-21", "radius": 10},
        #{"label": "Jiangxi_China_Flood", "lat": 29.0, "lon": 116.0, "date": "2020-07-12", "radius": 15},
        #{"label": "Kyushu_Japan_Flood", "lat": 32.8, "lon": 130.7, "date": "2020-07-04", "radius": 10},
        #{"label": "Jakarta_Indonesia_Flood", "lat": -6.2, "lon": 106.8, "date": "2020-01-02", "radius": 10},
        #{"label": "New_South_Wales_Australia", "lat": -33.8, "lon": 150.9, "date": "2021-03-22", "radius": 15},
        
        # Europe
        {"label": "Ahr_Valley_Germany_Flood", "lat": 50.5, "lon": 7.0, "date": "2021-07-15", "radius": 10},
        {"label": "Emilia_Romagna_Italy", "lat": 44.5, "lon": 11.3, "date": "2023-05-18", "radius": 15},
        {"label": "Thessaly_Greece_Flood", "lat": 39.5, "lon": 22.0, "date": "2023-09-07", "radius": 15},
        
        # Americas
        {"label": "Houston_Texas_Harvey", "lat": 29.7, "lon": -95.3, "date": "2017-08-27", "radius": 15},
        {"label": "Rio_Grande_Brazil", "lat": -30.0, "lon": -51.2, "date": "2024-05-05", "radius": 15},
        {"label": "British_Columbia_Canada", "lat": 49.0, "lon": -122.3, "date": "2021-11-16", "radius": 15},
        {"label": "Vermont_USA_Flood", "lat": 44.2, "lon": -72.5, "date": "2023-07-11", "radius": 10},
        
        # Africa
        {"label": "KwaZulu_Natal_South_Africa", "lat": -29.8, "lon": 31.0, "date": "2022-04-12", "radius": 10},
        {"label": "Khartoum_Sudan_Flood", "lat": 15.6, "lon": 32.5, "date": "2020-09-05", "radius": 15},
        {"label": "Nairobi_Kenya_Flood", "lat": -1.3, "lon": 36.8, "date": "2024-04-25", "radius": 10},
        {"label": "Beira_Mozambique_Idai", "lat": -19.8, "lon": 34.8, "date": "2019-03-15", "radius": 15}
    ]

    logging.info(f"Initiating bulk download for {len(flood_events)} locations.")
    results = {"success": [], "failed": []}

    for i, event in enumerate(flood_events):
        logging.info(f"--- Processing {i+1}/{len(flood_events)} ---")
        
        # Exponential backoff loop for rate limits
        max_retries = 3
        retry_delay = 5
        success = False
        
        for attempt in range(max_retries):
            success = fetch_fusion_ready_imagery(
                lat=event["lat"], lon=event["lon"], radius_km=event["radius"],
                target_date=event["date"], event_label=event["label"], output_dir=output_dir
            )
            if success:
                break
            elif attempt < max_retries - 1:
                logging.warning(f"Retrying {event['label']} in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 2 # Exponential increase: 5s, 10s, 20s
                
        if success:
            results["success"].append(event["label"])
        else:
            results["failed"].append(event["label"])
            
        # Base delay between locations to prevent throttling
        time.sleep(3) 

    # Final Report
    logging.info("\n=== Bulk Download Complete ===")
    logging.info(f"Total Attempted: {len(flood_events)}")
    logging.info(f"Successful: {len(results['success'])}")
    logging.info(f"Failed: {len(results['failed'])}")
    
    # Save failed list for easy retry later
    if results["failed"]:
        with open("failed_downloads.json", "w") as f:
            json.dump(results["failed"], f)
        logging.warning("Saved list of failed locations to failed_downloads.json")

if __name__ == "__main__":
    run_bulk_download()

# Revise Multiple Download imagery logic

In [ ]:
import os
import sys
import time
import logging
import gc
import pandas as pd
import numpy as np
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from requests.exceptions import ReadTimeout

# Output to stdout to prevent environmental red-text highlighting
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("bulk_download.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

# Canonical band order — strictly locked for U-Net multi-channel input
BAND_ORDER = ['SAR_CoPol', 'SAR_CrossPol', 'B02', 'B03', 'B04', 'B08', 'SCL']

def fetch_fusion_ready_imagery(lat, lon, radius_km, target_date, event_label, output_dir):
    target_dt     = pd.to_datetime(target_date, utc=True)
    degree_buffer = radius_km / 111.0
    bbox          = [lon - degree_buffer, lat - degree_buffer,
                     lon + degree_buffer, lat + degree_buffer]

    logging.info(f"Processing: {event_label} | Target: {target_date}")

    try:
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=planetary_computer.sign_inplace,
        )

        # EPSG:3857 guarantees square 10x10m pixels globally. 
        # Crucial for CNN spatial filters.
        shared_crs        = "EPSG:3857"
        shared_resolution = 10 

        # ── SENTINEL-1 ────────────────────────────────────────────────────────
        s1_range = (
            f"{(target_dt - pd.to_timedelta('7 days')).strftime('%Y-%m-%d')}/"
            f"{(target_dt + pd.to_timedelta('7 days')).strftime('%Y-%m-%d')}"
        )
        s1_items = list(catalog.search(
            collections=["sentinel-1-rtc"], bbox=bbox, datetime=s1_range
        ).items())

        if not s1_items:
            logging.error(f"[{event_label}] No S1 data in 14-day window. Skipping.")
            return False

        s1_items.sort(key=lambda x: abs(pd.to_datetime(x.datetime) - target_dt))
        best_s1 = s1_items[0]
        s1_delta = abs(pd.to_datetime(best_s1.datetime) - target_dt).days
        logging.info(f"[{event_label}] S1: {best_s1.datetime.date()} (delta={s1_delta}d)")

        assets = best_s1.assets.keys()
        if   'vv' in assets and 'vh' in assets: pol_bands = ['vv', 'vh']
        elif 'hh' in assets and 'hv' in assets: pol_bands = ['hh', 'hv']
        else:
            logging.error(f"[{event_label}] No dual-pol bands found.")
            return False

        s1_frame = odc.stac.load(
            [best_s1], bands=pol_bands, bbox=bbox,
            crs=shared_crs, resolution=shared_resolution,
            patch_url=planetary_computer.sign
        ).isel(time=0)

        # Physical calibration: linear -> dB, clip to [-30, 0] dB, normalize [0, 1]
        s1_valid = s1_frame.where(s1_frame > 0)
        s1_db    = 10 * np.log10(s1_valid)
        s1_norm  = ((s1_db + 30) / 30.0).clip(0, 1)
        s1_norm  = s1_norm.rename({pol_bands[0]: 'SAR_CoPol', pol_bands[1]: 'SAR_CrossPol'})

        # ── SENTINEL-2 ────────────────────────────────────────────────────────
        s2_range = (
            f"{(target_dt - pd.to_timedelta('30 days')).strftime('%Y-%m-%d')}/"
            f"{(target_dt + pd.to_timedelta('14 days')).strftime('%Y-%m-%d')}"
        )
        s2_items = list(catalog.search(
            collections=["sentinel-2-l2a"], bbox=bbox, datetime=s2_range,
            query={"eo:cloud_cover": {"lt": 30}}
        ).items())

        if s2_items:
            s2_items.sort(key=lambda x: abs(pd.to_datetime(x.datetime) - target_dt))
            best_s2    = s2_items[0]
            s2_delta   = abs(pd.to_datetime(best_s2.datetime) - target_dt).days
            s2_cloud   = best_s2.properties.get("eo:cloud_cover", "?")
            logging.info(f"[{event_label}] S2: {best_s2.datetime.date()} "
                         f"(delta={s2_delta}d, cloud={s2_cloud:.1f}%)")

            s2_frame = odc.stac.load(
                [best_s2], bands=["B02", "B03", "B04", "B08", "SCL"],
                bbox=bbox, crs=shared_crs, resolution=shared_resolution,
                patch_url=planetary_computer.sign
            ).isel(time=0)

            opt_bands = s2_frame[["B02", "B03", "B04", "B08"]]
            scl_band  = s2_frame[["SCL"]]

            # Mask nodata (-32768) and normalize reflectance to [0, 1]
            opt_valid = opt_bands.where((opt_bands > 0) & (opt_bands != -32768))
            opt_norm  = (opt_valid / 10000.0).clip(0, 1)

            # Reproject S2 onto S1 grid. 
            # Nearest neighbor for SCL prevents categorical corruption.
            opt_norm = opt_norm.rio.reproject_match(s1_norm)
            scl_band = scl_band.rio.reproject_match(s1_norm, resampling=Resampling.nearest)

            fused_ds = xr.merge([s1_norm, opt_norm, scl_band], compat="override")
            del s2_frame, opt_bands, opt_valid, opt_norm

        else:
            logging.warning(f"[{event_label}] No S2 <30% cloud in 44-day window. Padding NaN.")
            nan_template = xr.full_like(s1_norm['SAR_CoPol'], np.nan)
            fused_ds = xr.Dataset({
                'SAR_CoPol': s1_norm['SAR_CoPol'],
                'SAR_CrossPol': s1_norm['SAR_CrossPol'],
                'B02': nan_template, 'B03': nan_template, 'B04': nan_template, 
                'B08': nan_template, 'SCL': nan_template
            })

        # ── EXPORT ────────────────────────────────────────────────────────────
        # Enforce target band sequence before saving
        fused_ds = fused_ds[BAND_ORDER]
        fused_da = fused_ds.to_array(dim="band").astype("float32")
        fused_da = fused_da.assign_coords(band=BAND_ORDER)
        fused_da.rio.write_nodata(np.nan, inplace=True)
        fused_da.rio.write_crs(shared_crs, inplace=True)

        safe_label = event_label.replace(" ", "_").upper()
        safe_date  = target_date.replace("-", "")
        filename   = f"MANZAR_{safe_label}_S1S2_{safe_date}_{lat:.4f}_{lon:.4f}.tif"
        filepath   = os.path.join(output_dir, filename)

        fused_da.rio.to_raster(filepath, compress="LZW", tiled=True)
        logging.info(f"[{event_label}] SAVED: {filename}")

        del s1_frame, s1_valid, s1_db, s1_norm, fused_ds, fused_da
        gc.collect()
        return True

    except ReadTimeout:
        logging.error(f"[{event_label}] STAC API timeout.")
        return False
    except Exception as e:
        logging.error(f"[{event_label}] FAILED: {str(e)}")
        return False

def run_bulk_download():
    output_dir = "./data/raw_flood_stacks"
    os.makedirs(output_dir, exist_ok=True)

    flood_events = [
        # Pakistan
        {"label": "Sindh_Pakistan_Flood_1", "lat": 26.2, "lon": 68.0, "date": "2022-08-25", "radius": 15},
        {"label": "Sindh_Pakistan_Flood_2", "lat": 27.5, "lon": 68.5, "date": "2022-08-26", "radius": 15},
        {"label": "KPK_Pakistan_Flood", "lat": 34.0, "lon": 71.9, "date": "2022-08-28", "radius": 15},
        {"label": "Balochistan_Pakistan_Flood", "lat": 28.5, "lon": 66.0, "date": "2022-07-25", "radius": 15},
        {"label": "Punjab_Pakistan_Flood", "lat": 30.2, "lon": 70.9, "date": "2023-08-15", "radius": 15},
        
        # South Asia
        {"label": "Assam_India_Flood", "lat": 26.2, "lon": 92.0, "date": "2020-07-15", "radius": 15},
        {"label": "Kerala_India_Flood", "lat": 9.9, "lon": 76.2, "date": "2018-08-16", "radius": 15},
        {"label": "Sylhet_Bangladesh_Flood", "lat": 24.9, "lon": 91.8, "date": "2022-06-18", "radius": 15},
        {"label": "Kathmandu_Nepal_Flood", "lat": 27.7, "lon": 85.3, "date": "2017-08-12", "radius": 10},
        
        # Asia Pacific
        {"label": "Henan_China_Flood", "lat": 34.7, "lon": 113.6, "date": "2021-07-21", "radius": 10},
        {"label": "Jiangxi_China_Flood", "lat": 29.0, "lon": 116.0, "date": "2020-07-12", "radius": 15},
        {"label": "Kyushu_Japan_Flood", "lat": 32.8, "lon": 130.7, "date": "2020-07-04", "radius": 10},
        {"label": "Jakarta_Indonesia_Flood", "lat": -6.2, "lon": 106.8, "date": "2020-01-02", "radius": 10},
        {"label": "New_South_Wales_Australia", "lat": -33.8, "lon": 150.9, "date": "2021-03-22", "radius": 15},
        
        #{"label": "Ahr_Valley_Germany_Flood",  "lat": 50.5,  "lon":   7.0,  "date": "2021-07-15", "radius": 10},
        #{"label": "Emilia_Romagna_Italy",       "lat": 44.5,  "lon":  11.3,  "date": "2023-05-18", "radius": 15},
        #{"label": "Thessaly_Greece_Flood",      "lat": 39.5,  "lon":  22.0,  "date": "2023-09-07", "radius": 15},
        #{"label": "Houston_Texas_Harvey",       "lat": 29.7,  "lon": -95.3,  "date": "2017-08-27", "radius": 15},
        #{"label": "Rio_Grande_Brazil",          "lat": -30.0, "lon": -51.2,  "date": "2024-05-05", "radius": 15},
        #{"label": "British_Columbia_Canada",    "lat": 49.0,  "lon": -122.3, "date": "2021-11-16", "radius": 15},
        #{"label": "Vermont_USA_Flood",          "lat": 44.2,  "lon": -72.5,  "date": "2023-07-11", "radius": 10},
        #{"label": "KwaZulu_Natal_South_Africa", "lat": -29.8, "lon":  31.0,  "date": "2022-04-12", "radius": 10},
        #{"label": "Khartoum_Sudan_Flood",       "lat": 15.6,  "lon":  32.5,  "date": "2020-09-05", "radius": 15},
        #{"label": "Nairobi_Kenya_Flood",        "lat": -1.3,  "lon":  36.8,  "date": "2024-04-25", "radius": 10},
        #{"label": "Beira_Mozambique_Idai",      "lat": -19.8, "lon":  34.8,  "date": "2019-03-15", "radius": 15},
    ]

    logging.info(f"Starting bulk download: {len(flood_events)} events.")
    successful, failed = 0, 0

    for i, event in enumerate(flood_events):
        logging.info(f"--- [{i+1}/{len(flood_events)}] {event['label']} ---")
        max_retries, retry_delay, success = 3, 5, False

        for attempt in range(max_retries):
            success = fetch_fusion_ready_imagery(
                lat=event["lat"], lon=event["lon"], radius_km=event["radius"],
                target_date=event["date"], event_label=event["label"],
                output_dir=output_dir
            )
            if success:
                break
            elif attempt < max_retries - 1:
                logging.warning(f"Retry {attempt+2}/{max_retries} in {retry_delay}s...")
                time.sleep(retry_delay)
                retry_delay *= 2

        successful += success
        failed     += not success
        time.sleep(3)

    logging.info(f"\n=== Done: {successful} success, {failed} failed ===")

if __name__ == "__main__":
    run_bulk_download()

# Visulize the output

In [ ]:
import os
import numpy as np
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Band index map for 7-band stack
# [0] SAR_CoPol (VV or HH)
# [1] SAR_CrossPol (VH or HV)
# [2] B02 Blue
# [3] B03 Green
# [4] B04 Red
# [5] B08 NIR
# [6] SCL

def percentile_stretch(arr, lo=2, hi=98):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return np.zeros_like(arr)
    p_lo, p_hi = np.nanpercentile(valid, lo), np.nanpercentile(valid, hi)
    if p_hi == p_lo:
        return np.zeros_like(arr)
    return np.clip((arr - p_lo) / (p_hi - p_lo), 0, 1)


def make_sar_composite(da):
    vv = da.values[0].astype(np.float32)
    vh = da.values[1].astype(np.float32)
    vv_s = percentile_stretch(vv)
    vh_s = percentile_stretch(vh)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(vh > 0, vv / vh, np.nan)
    ratio_s = percentile_stretch(ratio)
    comp = np.dstack((vv_s, vh_s, ratio_s))
    comp[np.isnan(comp)] = 0
    return comp


def make_rgb(da, r_idx, g_idx, b_idx):
    r = percentile_stretch(da.values[r_idx].astype(np.float32))
    g = percentile_stretch(da.values[g_idx].astype(np.float32))
    b = percentile_stretch(da.values[b_idx].astype(np.float32))
    rgb = np.dstack((r, g, b))
    rgb[np.isnan(rgb)] = 0
    return rgb


def make_ndwi(da):
    green = da.values[3].astype(np.float32)
    nir   = da.values[5].astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        ndwi = np.where(
            (~np.isnan(green)) & (~np.isnan(nir)) & (green + nir > 0),
            (green - nir) / (green + nir),
            np.nan
        )
    return ndwi


def check_optical(da):
    """Returns (has_optical, nan_pct). Safe for any band count."""
    if da.shape[0] < 5:
        return False, 100.0
    b04 = da.values[4].astype(np.float32)
    nan_pct = np.sum(np.isnan(b04)) / b04.size * 100
    return nan_pct < 70.0, nan_pct


def visualize_flood_stack(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return

    da = rioxarray.open_rasterio(filepath)
    fname = os.path.basename(filepath)
    n_bands = da.shape[0]

    print(f"File    : {fname}")
    print(f"Shape   : {da.values.shape}  (bands, height, width)")
    print(f"CRS     : {da.rio.crs}")
    print(f"NoData  : {da.rio.nodata}")
    print(f"Bands   : {n_bands}")

    has_sar     = n_bands >= 2
    optical_ok, nan_pct = check_optical(da)

    if not has_sar:
        print("ERROR: Need at least 2 SAR bands. Skipping.")
        da.close()
        return

    if not optical_ok:
        print(f"WARNING: Optical NaN={nan_pct:.1f}% — SAR panels only.")

    n_cols = 4 if optical_ok else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 7),
                             gridspec_kw={'wspace': 0.05})
    if n_cols == 1:
        axes = [axes]

    fig.suptitle(fname, fontsize=10, fontweight='bold')

    col = 0

    # SAR grayscale (VV/HH)
    vv_s = percentile_stretch(da.values[0].astype(np.float32))
    vv_s[np.isnan(vv_s)] = 0
    axes[col].imshow(vv_s, cmap='gray', interpolation='nearest')
    axes[col].set_title("SAR CoPol\n(Grayscale)", fontsize=11, fontweight='bold')
    axes[col].axis('off')
    col += 1

    # SAR false color composite
    sar_comp = make_sar_composite(da)
    axes[col].imshow(sar_comp, interpolation='nearest')
    axes[col].set_title("SAR False Color\n(R=VV  G=VH  B=VV/VH)", fontsize=11, fontweight='bold')
    axes[col].axis('off')
    col += 1

    if optical_ok:
        # S2 true color
        rgb = make_rgb(da, r_idx=4, g_idx=3, b_idx=2)
        axes[col].imshow(rgb, interpolation='nearest')
        axes[col].set_title("S2 True Color\n(R=B04  G=B03  B=B02)", fontsize=11, fontweight='bold')
        axes[col].axis('off')
        col += 1

        # NDWI
        ndwi = make_ndwi(da)
        im = axes[col].imshow(ndwi, cmap='RdYlBu', vmin=-0.5, vmax=0.5,
                              interpolation='nearest')
        plt.colorbar(im, ax=axes[col], fraction=0.03, pad=0.02)
        axes[col].set_title("NDWI\n(Blue = water)", fontsize=11, fontweight='bold')
        axes[col].axis('off')

    plt.show()
    plt.close()
    da.close()


def visualize_all(directory):
    from pathlib import Path
    tif_files = sorted(Path(directory).glob("*.tif"))
    if not tif_files:
        print(f"No .tif files in {directory}")
        return
    print(f"Visualizing {len(tif_files)} files...\n")
    for fp in tif_files:
        visualize_flood_stack(str(fp))
        print()



# Single file
#visualize_flood_stack(
#    "/kaggle/working/data/raw_flood_stacks/MANZAR_RIO_GRANDE_BRAZIL_S1S2_20240505_-30.0000_-51.2000.tif"
#)

# Or all files
visualize_all("/kaggle/working/data/raw_flood_stacks")

# Quality Checker for all images

In [ ]:
import os
import rioxarray
import numpy as np
import pandas as pd
from pathlib import Path

# Canonical band order — must match download script's BAND_ORDER
# [0] SAR_CoPol  [1] SAR_CrossPol  [2] B02  [3] B03  [4] B04  [5] B08  [6] SCL
BAND_NAMES = [
    "SAR_CoPol",
    "SAR_CrossPol",
    "B02 (Blue)",
    "B03 (Green)",
    "B04 (Red)",
    "B08 (NIR)",
    "SCL (Mask)",
]

# SCL integer class definitions (ESA standard)
SCL_CLASSES = {
    0:  "No Data",
    1:  "Saturated/Defective",
    2:  "Dark Area",
    3:  "Cloud Shadow",
    4:  "Vegetation",
    5:  "Bare Soil",
    6:  "Water",
    7:  "Unclassified",
    8:  "Cloud Medium Prob",
    9:  "Cloud High Prob",
    10: "Thin Cirrus",
    11: "Snow/Ice",
}


def _band_stats(arr, total_pixels):
    """Compute per-band stats. Returns dict."""
    data = arr.astype(np.float64)
    nans = np.isnan(data)
    infs = np.isinf(data)
    valid = data[~nans & ~infs]
    nan_pct  = np.sum(nans) / total_pixels * 100
    inf_pct  = np.sum(infs) / total_pixels * 100
    zero_pct = np.sum(data == 0) / total_pixels * 100

    if len(valid) == 0:
        return {
            "Min": np.nan, "Max": np.nan, "Mean": np.nan, "Std": np.nan,
            "NaN%": nan_pct, "Inf%": inf_pct, "Zero%": zero_pct
        }
    return {
        "Min":   float(np.min(valid)),
        "Max":   float(np.max(valid)),
        "Mean":  float(np.mean(valid)),
        "Std":   float(np.std(valid)),
        "NaN%":  nan_pct,
        "Inf%":  inf_pct,
        "Zero%": zero_pct,
    }


def _parse_scl(scl_arr, total_pixels):
    """
    Parse SCL band into per-class percentages.
    Detects if SCL was accidentally normalized to [0,1] and recovers integer classes.
    Returns (class_pct dict, was_normalized bool).
    """
    data   = scl_arr.astype(np.float64)
    valid  = data[~np.isnan(data)]

    if len(valid) == 0:
        return {}, False, "SCL entirely NaN"

    max_val = float(np.nanmax(valid))

    # If max <= 1.0 and values aren't whole numbers → SCL was normalized
    was_normalized = (max_val <= 1.0) and not np.all(valid == np.round(valid))
    warning = None

    if was_normalized:
        # Recover: undo /10000 normalization
        valid = np.round(valid * 10000).astype(int)
        warning = "SCL_NORMALIZED: SCL was divided by 10000 — old tile. Re-download recommended."
    else:
        valid = np.round(valid).astype(int)
        # Clamp to valid SCL range
        valid = valid[(valid >= 0) & (valid <= 11)]

    unique, counts = np.unique(valid, return_counts=True)
    class_pct = {int(k): float(v / total_pixels * 100) for k, v in zip(unique, counts)}
    return class_pct, was_normalized, warning


def dl_preflight_qa(filepath):
    try:
        da = rioxarray.open_rasterio(filepath)
    except Exception as e:
        return None, None, [f"CRITICAL: Failed to load — {e}"]

    n_bands, height, width = da.shape
    total_pixels = height * width

    if n_bands != len(BAND_NAMES):
        da.close()
        return None, None, [f"CRITICAL: Expected {len(BAND_NAMES)} bands, found {n_bands}."]

    # ── Per-band statistics ────────────────────────────────────────────────
    rows = []
    stats_by_band = {}
    for i, name in enumerate(BAND_NAMES):
        s = _band_stats(da.values[i], total_pixels)
        stats_by_band[name] = s
        rows.append({
            "Band":   name,
            "Min":    f"{s['Min']:.4f}"  if not np.isnan(s['Min'])  else "NaN",
            "Max":    f"{s['Max']:.4f}"  if not np.isnan(s['Max'])  else "NaN",
            "Mean":   f"{s['Mean']:.4f}" if not np.isnan(s['Mean']) else "NaN",
            "Std":    f"{s['Std']:.4f}"  if not np.isnan(s['Std'])  else "NaN",
            "NaN%":   f"{s['NaN%']:.2f}%",
            "Inf%":   f"{s['Inf%']:.2f}%",
            "Zero%":  f"{s['Zero%']:.2f}%",
        })
    df = pd.DataFrame(rows)

    # ── SCL analysis ───────────────────────────────────────────────────────
    class_pct, scl_normalized, scl_warning = _parse_scl(da.values[6], total_pixels)

    cloud_pct  = class_pct.get(8, 0.0) + class_pct.get(9, 0.0) + class_pct.get(10, 0.0)
    shadow_pct = class_pct.get(3, 0.0)
    water_pct  = class_pct.get(6, 0.0)
    nodata_pct = class_pct.get(0, 0.0)

    scl_summary = {
        "cloud":  cloud_pct,
        "shadow": shadow_pct,
        "water":  water_pct,
        "nodata": nodata_pct,
        "classes": class_pct,
    }

    # ── Flag logic ─────────────────────────────────────────────────────────
    flags = []

    # SCL corruption
    if scl_warning:
        flags.append(scl_warning)

    # Band-level NaN — check all bands, threshold at 50%
    for name in BAND_NAMES:
        nan_pct = stats_by_band[name]["NaN%"]
        if nan_pct > 50:
            flags.append(f"HIGH_NAN [{name}]: {nan_pct:.1f}%")
        elif nan_pct > 20:
            flags.append(f"PARTIAL_NAN [{name}]: {nan_pct:.1f}%")

    # SAR-specific: check for zero-dominated bands (nodata written as 0, not NaN)
    for sar_band in ["SAR_CoPol", "SAR_CrossPol"]:
        zero_pct = stats_by_band[sar_band]["Zero%"]
        if zero_pct > 50:
            flags.append(f"SAR_ZERO_DOMINATED [{sar_band}]: {zero_pct:.1f}% zeros — nodata not NaN")

    # SAR value range sanity (after dB normalization to [0,1])
    for sar_band in ["SAR_CoPol", "SAR_CrossPol"]:
        s = stats_by_band[sar_band]
        if not np.isnan(s["Max"]):
            if s["Max"] > 10:
                flags.append(f"SAR_NOT_NORMALIZED [{sar_band}]: max={s['Max']:.1f} — still in raw dB or linear scale")
            if s["Min"] < 0:
                flags.append(f"SAR_NEGATIVE [{sar_band}]: min={s['Min']:.4f} — unexpected")

    # Optical value range sanity (after /10000 normalization to [0,1])
    for opt_band in ["B02 (Blue)", "B03 (Green)", "B04 (Red)", "B08 (NIR)"]:
        s = stats_by_band[opt_band]
        if not np.isnan(s["Min"]) and s["Min"] < 0:
            flags.append(f"OPTICAL_NEGATIVE [{opt_band}]: min={s['Min']:.4f} — nodata bleed (-32768 not masked)")
        if not np.isnan(s["Max"]) and s["Max"] > 1.01:
            flags.append(f"OPTICAL_OVERFLOW [{opt_band}]: max={s['Max']:.4f} — not normalized to [0,1]")

    # Scale mismatch between optical and SAR (catches old un-normalized tiles)
    nir_max = stats_by_band["B08 (NIR)"]["Max"]
    vv_max  = stats_by_band["SAR_CoPol"]["Max"]
    if not np.isnan(nir_max) and not np.isnan(vv_max):
        if nir_max > 100 and vv_max < 10:
            flags.append(f"SCALE_MISMATCH: optical_max={nir_max:.0f} vs SAR_max={vv_max:.2f}")

    # SCL-based flags
    if water_pct == 0 and not scl_normalized:
        flags.append("NO_WATER_IN_SCL: SCL class 6 absent — flood signal may not be visible in optical")
    if cloud_pct > 30:
        flags.append(f"HIGH_CLOUD: {cloud_pct:.1f}% — optical bands likely corrupted")
    if shadow_pct > 20:
        flags.append(f"HIGH_SHADOW: {shadow_pct:.1f}%")
    if nodata_pct > 30:
        flags.append(f"HIGH_SCL_NODATA: {nodata_pct:.1f}% — tile boundary or missing acquisition")

    # Footprint mismatch: optical and SAR NaN% differ significantly
    sar_nan  = stats_by_band["SAR_CoPol"]["NaN%"]
    opt_nan  = stats_by_band["B04 (Red)"]["NaN%"]
    if abs(sar_nan - opt_nan) > 5 and not (np.isnan(sar_nan) or np.isnan(opt_nan)):
        flags.append(
            f"FOOTPRINT_MISMATCH: SAR_NaN={sar_nan:.1f}% vs B04_NaN={opt_nan:.1f}% "
            f"— S1/S2 grids not aligned"
        )

    da.close()
    return df, scl_summary, flags


def run_batch_qa(directory):
    tif_files = sorted(Path(directory).glob("*.tif"))

    if not tif_files:
        print(f"No .tif files found in {directory}")
        return

    print(f"BATCH QA  |  {len(tif_files)} files  |  {directory}\n")

    clean, flagged, failed = [], [], []

    for idx, filepath in enumerate(tif_files, 1):
        fname = filepath.name
        print(f"{'='*72}")
        print(f"[{idx}/{len(tif_files)}] {fname}")
        print(f"{'='*72}")

        band_df, scl, flags = dl_preflight_qa(filepath)

        if band_df is None:
            print(f"  ERROR: {flags[0]}")
            failed.append(fname)
            print()
            continue

        # Band stats table
        print(band_df.to_string(index=False))

        # SCL summary
        print(f"\nSCL Summary:")
        print(f"  Cloud (8+9+10): {scl['cloud']:.2f}%  |  "
              f"Shadow (3): {scl['shadow']:.2f}%  |  "
              f"Water (6): {scl['water']:.2f}%  |  "
              f"NoData (0): {scl['nodata']:.2f}%")

        # Non-zero SCL classes
        active = {SCL_CLASSES.get(k, f"Class {k}"): f"{v:.2f}%"
                  for k, v in sorted(scl['classes'].items()) if v > 0.5}
        if active:
            print(f"  Active classes: {active}")

        # Flags
        if flags:
            print(f"\nRED FLAGS ({len(flags)}):")
            for f in flags:
                print(f"  -> {f}")
            flagged.append(fname)
        else:
            print(f"\nSTATUS: CLEAN")
            clean.append(fname)

        print()

    # Summary
    print(f"{'='*72}")
    print(f"DONE  |  Total: {len(tif_files)}  |  "
          f"Clean: {len(clean)}  |  Flagged: {len(flagged)}  |  Failed: {len(failed)}")

    if clean:
        print(f"\nCLEAN ({len(clean)}):")
        for f in clean:
            print(f"  {f}")
    if flagged:
        print(f"\nFLAGGED ({len(flagged)}):")
        for f in flagged:
            print(f"  {f}")
    if failed:
        print(f"\nFAILED ({len(failed)}):")
        for f in failed:
            print(f"  {f}")


run_batch_qa("/kaggle/working/data/raw_flood_stacks")

# Delete files

In [ ]:

import os, glob
for f in glob.glob("/kaggle/working/data/raw_flood_stacks/*.tif"):
    os.remove(f)
print("Cleared.")